# Create computational mesh (bluemesh2d)

Build a gradient-limited element-size field from GEBCO bathymetry and generate
the SnapWave unstructured mesh for the case AoI.

**Prerequisites:** ``01_Generate_tif_From_GEBCO.ipynb`` (topobathy GeoTIFF and AoI).

## Configuration


> **TODO — adapt to bluemesh2d.** The vendored mesher moved from
> `pymesh2d` to `bluemesh2d` and the API evolved. Pending updates in
> the code cells below:
>
> 1. Imports: `from pymesh2d....` → `from bluemesh2d....`
> 2. `getiso_polygone` was renamed to `getiso_polygon` (`geom_util.getiso`)
> 3. `clean_polygon` no longer exists — see
>    `bluemesh2d.geomesh_util.water_polygon.extract_water_polygon`
> 4. `mesh_util.circo.small_flow_centers` no longer exists — see the new
>    `ortho_merge` / `merge_circumcenters` / `smood` workflow
>    (and the higher-level `bluemesh2d.pipeline` / `bluemesh2d.meshgen`)


In [ ]:
import rioxarray

from utils.config import PATHS, resolve

topobathy = rioxarray.open_rasterio(resolve(PATHS["topobathy_file"]))
topobathy

In [ ]:
topobathy.plot(vmin=-200, vmax=0, cmap="terrain")

In [ ]:
import pyproj
from pymesh2d.geom_util.proj_util import get_local_utm_crs

topobathy_crs = pyproj.CRS.from_epsg(4326)
utm_crs = get_local_utm_crs(topobathy_crs, topobathy.x, topobathy.y)

topobathy_crs, utm_crs

In [ ]:
import geopandas as gpd

aoi_polygon = gpd.read_file(resolve(PATHS["aoi_file"])).loc[0, "geometry"]
aoi_polygon

In [ ]:
from pymesh2d.geom_util.getiso import getiso_polygone
from shapely import make_valid
from shapely.geometry import MultiPolygon

# Extract coastline (ocean area)
polygon_cost = getiso_polygone(
    topobathy.x, topobathy.y, topobathy.squeeze().values, zmax=0.0
)
# Extract the area of the domain
polygon_indef = getiso_polygone(
    topobathy.x, topobathy.y, topobathy.squeeze().values, zmax=5000
)

# Make the polygons valid
cost_mp = make_valid(MultiPolygon(polygon_cost)).buffer(0)
indef_mp = make_valid(MultiPolygon(polygon_indef)).buffer(0)

# Intersect the polygons to get the domain
topobathy_dominio = make_valid(cost_mp.intersection(indef_mp)).buffer(0)
# topobathy_dominio = max(topobathy_dominio_prev.geoms, key=lambda p: p.area)

topobathy_dominio

In [ ]:
aoi_topobathy_polygon = aoi_polygon.intersection(topobathy_dominio)

# export aoi_topobathy_plygon
gdf = gpd.GeoDataFrame(geometry=[aoi_topobathy_polygon], crs=topobathy_crs)
# gdf.to_file("inputs/AoI_Polygons/AoI_topobathy.shp")

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": ccrs.PlateCarree()})

topobathy.plot(vmin=-200, vmax=0, cmap="terrain", ax=ax)
gdf.plot(ax=ax, color="red", alpha=0.25, edgecolor="red")

In [ ]:
import numpy as np
from pymesh2d.geomesh_util.depth_field import depth_field_from_tif
from utils.mesh.mesh_sizing import plot_hfun, precalc_hfuns

depth_field = depth_field_from_tif(str(resolve(PATHS["topobathy_file"])), utm_crs)


def hfun_depth_quadratic(test, depth_field, hmin=np.inf, hmax=np.inf):
    test = np.asarray(test, dtype=float)
    if test.ndim == 1:
        test = test.reshape(1, -1)

    h = depth_field(test) if callable(depth_field) else np.asarray(depth_field)
    if np.size(h) == 1:
        h_scalar = float(np.asarray(h).reshape(-1)[0])
        h = np.full(test.shape[0], h_scalar)
    else:
        h = np.asarray(h).reshape(-1)
        if h.shape[0] != test.shape[0]:
            raise ValueError("depth_field size must match test points")

    hfun_values = 0.14 * h**2 + 28 * h + 655
    return np.clip(hfun_values, hmin, hmax)


def hfun(test):
    return hfun_depth_quadratic(test, depth_field, hmin=100, hmax=5000)


hfuns = precalc_hfuns(
    hfun, work_crs=utm_crs, lon=topobathy.x, lat=topobathy.y, geo_crs=topobathy_crs
)
plot_hfun(hfuns, hfun);


In [ ]:
from pymesh2d.geom_util.proj_util import reproject_geometry

domain_UTM = reproject_geometry(aoi_topobathy_polygon, topobathy_crs, utm_crs)

if domain_UTM.geom_type == "MultiPolygon":
    largest_polygon_domain_UTM = max(domain_UTM.geoms, key=lambda p: p.area)
else:
    largest_polygon_domain_UTM = domain_UTM

In [ ]:
from pymesh2d.geom_util.poly_util import polygon_to_node_edge, resample_polygon_hfun
from pymesh2d.geomesh_util.border_util import clean_polygon

# Use smoothed sizing function directly
poly_comput_UTM = resample_polygon_hfun(largest_polygon_domain_UTM, hfuns)

if poly_comput_UTM.geom_type == "MultiPolygon":
    poly_comput_UTM = max(poly_comput_UTM.geoms, key=lambda p: p.area)

poly_comput_UTM_clean = clean_polygon(poly_comput_UTM, min_vertices=15)

# Convert to nodes and edges
node_c, edge_c = polygon_to_node_edge(poly_comput_UTM_clean)

In [ ]:
from shapely.plotting import plot_polygon

fig, ax = plt.subplots(figsize=(8, 8))
plot_polygon(poly_comput_UTM_clean, ax=ax, facecolor="lightblue", alpha=0.2)

for e in edge_c.astype(int):
    ax.plot(node_c[e, 0], node_c[e, 1], "k-", lw=0.5)
ax.scatter(node_c[:, 0], node_c[:, 1], color="red", s=2)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title(f"Computational Domain: {len(node_c)} nodes")
ax.axis("equal");

In [ ]:
from pymesh2d.refine import refine

# Mesh generation options
opts_refine = {"kind": "delaunay"}

# Generate and smooth computational mesh
vert_c, etri_c, tria_c, tnum_c = refine(node_c, edge_c, [], opts_refine, hfuns)

In [ ]:
from pymesh2d.smooth import smooth

opts_smooth = {}

vert_c, etri_c, tria_c, tnum_c = smooth(
    vert_c, etri_c, tria_c, tnum_c, opts_smooth, hfuns
)

In [ ]:
from pymesh2d.mesh_util.circo import small_flow_centers
from pymesh2d.mesh_util.tricon import tricon

# Compute triangle connectivity and identify small flow links
edge_cc, _ = tricon(tria_c, None)

circumcenters, nlinktoosmall = small_flow_centers(
    vert_c, tria_c, edge_cc, removesmalllinkstrsh=0.11, conn=None
)

# Visualize both meshes
fig, ax = plt.subplots(figsize=(8, 8))

ax.triplot(
    vert_c[:, 0],
    vert_c[:, 1],
    tria_c[:, 0:3],
    color="steelblue",
    lw=0.5,
    label="Computational mesh",
)

if nlinktoosmall > 0:
    ax.scatter(
        circumcenters[:, 0],
        circumcenters[:, 1],
        color="red",
        s=20,
        alpha=0.7,
        zorder=5,
        label=f"Small flow links ({nlinktoosmall})",
    )
else:
    ax.text(
        0.02,
        0.98,
        "No small flow links detected",
        transform=ax.transAxes,
        fontsize=12,
        color="green",
        verticalalignment="top",
    )

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Nested Mesh System (UTM coordinates)")
ax.legend()
ax.axis("equal");

In [ ]:
from pymesh2d.geom_util.proj_util import reproject_node

# from pymesh2d.smood import smood

vvert_c = reproject_node(
    vert_c, get_local_utm_crs(topobathy_crs, topobathy.x, topobathy.y), topobathy_crs
)

# vert_2, etri_2, tria_2, tnum_2 = smood(
#     vvert_c,
#     etri_c,
#     tria_c,
#     tnum_c,
#     opts={
#         "strategy": "continuous",
#         "require_both_criteria": True,
#         "iter": 24,
#         "max_adapt_steps": 12,
#         "continuous_stagnation_patience": 2,
#         "preserve_merged_quads": True,
#     },
# )

In [ ]:
from pymesh2d.geomesh_util.interpolation_mesh import interpolate_from_tiff

z = interpolate_from_tiff(
    str(resolve(PATHS["topobathy_file"])), vvert_c, input_crs=topobathy_crs
)

In [ ]:
from pymesh2d.geomesh_util.grd_util import adcirc2DFlowFM

ds_out = adcirc2DFlowFM(np.column_stack((vvert_c, z)), tria_c)

In [ ]:
ds_out.to_netcdf(resolve(PATHS["mesh_file"]))